In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../data/DataCoSupplyChain_Cleaned.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (180519, 45)


In [3]:
# ============================================
# 2. BASIC DATASET OVERVIEW
# ============================================

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Rows: 180519
Columns: 45

Column Names:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Id', 'Customer Segment', 'Customer State', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Product Card Id', 'Product Category Id', 'Product Name', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'Shipping Mode']

Data Types:
Type                                 str
Days for shipping (real)          

In [4]:
# ============================================
# 3. PREPARE DATE COLUMNS
# ============================================

df["order date (DateOrders)"] = pd.to_datetime(
    df["order date (DateOrders)"],
    errors="coerce"
)

df["shipping date (DateOrders)"] = pd.to_datetime(
    df["shipping date (DateOrders)"],
    errors="coerce"
)

print("Order date range:")
print(df["order date (DateOrders)"].min(), "to", df["order date (DateOrders)"].max())

print("\nShipping date range:")
print(df["shipping date (DateOrders)"].min(), "to", df["shipping date (DateOrders)"].max())

print("\nMissing dates:")
print(df[["order date (DateOrders)", "shipping date (DateOrders)"]].isna().sum())

Order date range:
2015-01-01 00:00:00 to 2018-01-31 23:38:00

Shipping date range:
2015-01-03 00:00:00 to 2018-02-06 22:14:00

Missing dates:
order date (DateOrders)       0
shipping date (DateOrders)    0
dtype: int64


In [5]:
# ============================================
# 4. OVERALL BUSINESS PERFORMANCE
# ============================================

total_orders = df["Order Id"].nunique()
total_order_items = df["Order Item Id"].nunique()
total_sales = df["Sales"].sum()
total_profit = df["Order Profit Per Order"].sum()
avg_order_value = df.groupby("Order Id")["Sales"].sum().mean()
late_orders = df["Late_delivery_risk"].sum()
late_delivery_rate = (late_orders / len(df)) * 100

print("===== SUPPLY CHAIN PERFORMANCE =====")
print(f"Total Orders: {total_orders:,}")
print(f"Total Order Items: {total_order_items:,}")
print(f"Total Sales: ${total_sales:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Average Order Value: ${avg_order_value:,.2f}")
print(f"Late Delivery Risk: {late_orders:,}")
print(f"Late Delivery Risk Rate: {late_delivery_rate:.2f}%")

===== SUPPLY CHAIN PERFORMANCE =====
Total Orders: 65,752
Total Order Items: 180,519
Total Sales: $36,784,735.01
Total Profit: $3,966,902.97
Average Order Value: $559.45
Late Delivery Risk: 98,977
Late Delivery Risk Rate: 54.83%


In [6]:
# ============================================
# 5. MONTHLY SALES & LATE DELIVERY RISK
# ============================================

df["Order Month"] = df["order date (DateOrders)"].dt.to_period("M")

monthly_analysis = df.groupby("Order Month").agg(
    Sales=("Sales", "sum"),
    Late_Risk=("Late_delivery_risk", "sum"),
    Order_Items=("Order Item Id", "count")
).reset_index()

monthly_analysis["Late_Risk_Rate"] = (
    monthly_analysis["Late_Risk"] /
    monthly_analysis["Order_Items"] * 100
)

monthly_analysis["Order Month"] = monthly_analysis["Order Month"].astype(str)

print(monthly_analysis.head())
print("\nNumber of months:", len(monthly_analysis))

  Order Month         Sales  Late_Risk  Order_Items  Late_Risk_Rate
0     2015-01  1.051590e+06       2880         5322       54.114994
1     2015-02  9.270099e+05       2594         4729       54.853034
2     2015-03  1.051254e+06       2936         5362       54.755688
3     2015-04  1.014463e+06       2760         5126       53.843153
4     2015-05  1.050478e+06       2951         5357       55.086802

Number of months: 37


In [7]:
# ============================================
# 6. HIGHEST LATE-DELIVERY RISK MONTHS
# ============================================

top_risk_months = monthly_analysis.sort_values(
    "Late_Risk_Rate",
    ascending=False
).head(10)

print("TOP 10 HIGHEST-RISK MONTHS")
print(
    top_risk_months[
        ["Order Month", "Late_Risk_Rate", "Late_Risk", "Order_Items"]
    ].to_string(index=False)
)

TOP 10 HIGHEST-RISK MONTHS
Order Month  Late_Risk_Rate  Late_Risk  Order_Items
    2016-06       56.846063       2873         5054
    2015-09       56.692607       2914         5140
    2017-08       56.437323       2994         5305
    2018-01       56.288271       1195         2123
    2016-12       56.063769       2954         5269
    2016-03       55.758157       2905         5210
    2015-08       55.679879       2936         5273
    2016-05       55.658242       2951         5302
    2016-01       55.576453       2955         5317
    2015-07       55.463295       2939         5299


In [8]:
# ============================================
# 7. LATE-DELIVERY RISK BY SHIPPING MODE
# ============================================

shipping_analysis = df.groupby("Shipping Mode").agg(
    Order_Items=("Order Item Id", "count"),
    Late_Risk=("Late_delivery_risk", "sum")
).reset_index()

shipping_analysis["Late_Risk_Rate"] = (
    shipping_analysis["Late_Risk"] /
    shipping_analysis["Order_Items"] * 100
)

shipping_analysis = shipping_analysis.sort_values(
    "Late_Risk_Rate",
    ascending=False
)

print(shipping_analysis.to_string(index=False))

 Shipping Mode  Order_Items  Late_Risk  Late_Risk_Rate
   First Class        27814      26513       95.322499
  Second Class        35216      26987       76.632781
      Same Day         9737       4454       45.743042
Standard Class       107752      41023       38.071683


In [9]:
# ============================================
# 8. SHIPPING MODE VS SCHEDULED SHIPPING TIME
# ============================================

shipping_schedule = df.groupby("Shipping Mode").agg(
    Avg_Scheduled_Days=("Days for shipment (scheduled)", "mean"),
    Avg_Actual_Days=("Days for shipping (real)", "mean"),
    Late_Risk_Rate=("Late_delivery_risk", "mean")
).reset_index()

shipping_schedule["Late_Risk_Rate"] = (
    shipping_schedule["Late_Risk_Rate"] * 100
)

shipping_schedule = shipping_schedule.sort_values(
    "Late_Risk_Rate",
    ascending=False
)

print(shipping_schedule.to_string(index=False))

 Shipping Mode  Avg_Scheduled_Days  Avg_Actual_Days  Late_Risk_Rate
   First Class                 1.0         2.000000       95.322499
  Second Class                 2.0         3.990828       76.632781
      Same Day                 0.0         0.478279       45.743042
Standard Class                 4.0         3.995907       38.071683


In [10]:
# ============================================
# 9. LATE-DELIVERY RISK BY REGION
# ============================================

region_analysis = df.groupby("Order Region").agg(
    Order_Items=("Order Item Id", "count"),
    Late_Risk=("Late_delivery_risk", "sum")
).reset_index()

region_analysis["Late_Risk_Rate"] = (
    region_analysis["Late_Risk"] /
    region_analysis["Order_Items"] * 100
)

region_analysis = region_analysis.sort_values(
    "Late_Risk_Rate",
    ascending=False
)

print(region_analysis.to_string(index=False))

   Order Region  Order_Items  Late_Risk  Late_Risk_Rate
 Central Africa         1677        972       57.960644
     South Asia         7731       4350       56.266977
    East Africa         1852       1036       55.939525
 Western Europe        27109      15140       55.848611
 South of  USA          4045       2256       55.772559
 Eastern Europe         3920       2182       55.663265
    East of USA         6915       3849       55.661605
 Southeast Asia         9539       5297       55.529930
   Central Asia          553        306       55.334539
      West Asia         6009       3322       55.283741
     US Center          5887       3252       55.240360
Central America        28341      15518       54.754596
   North Africa         3232       1762       54.517327
Southern Europe         9431       5129       54.384477
   Eastern Asia         7280       3955       54.326923
  South America        14935       8111       54.308671
Northern Europe         9792       5292       54

In [11]:
# ============================================
# 10. SHIPPING MODE + REGION RISK
# ============================================

mode_region_analysis = df.groupby(
    ["Order Region", "Shipping Mode"]
).agg(
    Order_Items=("Order Item Id", "count"),
    Late_Risk=("Late_delivery_risk", "sum")
).reset_index()

mode_region_analysis["Late_Risk_Rate"] = (
    mode_region_analysis["Late_Risk"] /
    mode_region_analysis["Order_Items"] * 100
)

# Keep combinations with at least 100 order items
mode_region_analysis = mode_region_analysis[
    mode_region_analysis["Order_Items"] >= 100
]

mode_region_analysis = mode_region_analysis.sort_values(
    "Late_Risk_Rate",
    ascending=False
)

print(
    mode_region_analysis[
        ["Order Region", "Shipping Mode",
         "Order_Items", "Late_Risk_Rate"]
    ].head(20).to_string(index=False)
)

   Order Region Shipping Mode  Order_Items  Late_Risk_Rate
    East Africa   First Class          280       98.928571
Southern Africa   First Class          198       98.484848
    West Africa   First Class          463       97.840173
Southern Europe   First Class         1504       97.207447
   North Africa   First Class          486       97.119342
     South Asia   First Class         1283       96.648480
     US Center    First Class          986       96.450304
    East of USA   First Class         1089       96.418733
 Eastern Europe   First Class          658       96.048632
Northern Europe   First Class         1418       95.980254
Central America   First Class         4382       95.504336
 Southeast Asia   First Class         1552       95.489691
      West Asia   First Class          994       95.271630
 Western Europe   First Class         4312       95.245826
 South of  USA    First Class          600       95.166667
      Caribbean   First Class         1193       94.8868

In [12]:
# ============================================
# 11. SHIPPING DELAY GAP ANALYSIS
# ============================================

df["Shipping_Delay_Gap"] = (
    df["Days for shipping (real)"] -
    df["Days for shipment (scheduled)"]
)

delay_gap_analysis = df.groupby("Shipping Mode").agg(
    Order_Items=("Order Item Id", "count"),
    Avg_Delay_Gap=("Shipping_Delay_Gap", "mean"),
    Max_Delay_Gap=("Shipping_Delay_Gap", "max"),
    Late_Risk_Rate=("Late_delivery_risk", "mean")
).reset_index()

delay_gap_analysis["Late_Risk_Rate"] = (
    delay_gap_analysis["Late_Risk_Rate"] * 100
)

delay_gap_analysis = delay_gap_analysis.sort_values(
    "Avg_Delay_Gap",
    ascending=False
)

print(delay_gap_analysis.to_string(index=False))

 Shipping Mode  Order_Items  Avg_Delay_Gap  Max_Delay_Gap  Late_Risk_Rate
  Second Class        35216       1.990828              4       76.632781
   First Class        27814       1.000000              1       95.322499
      Same Day         9737       0.478279              1       45.743042
Standard Class       107752      -0.004093              2       38.071683


In [13]:
# ============================================
# 12. DEPARTMENT PERFORMANCE & DELIVERY RISK
# ============================================

department_analysis = df.groupby("Department Name").agg(
    Sales=("Sales", "sum"),
    Profit=("Order Profit Per Order", "sum"),
    Order_Items=("Order Item Id", "count"),
    Late_Risk=("Late_delivery_risk", "sum")
).reset_index()

department_analysis["Late_Risk_Rate"] = (
    department_analysis["Late_Risk"] /
    department_analysis["Order_Items"] * 100
)

department_analysis = department_analysis.sort_values(
    "Sales",
    ascending=False
)

print(
    department_analysis[
        ["Department Name", "Sales", "Profit",
         "Order_Items", "Late_Risk_Rate"]
    ].to_string(index=False)
)

   Department Name        Sales       Profit  Order_Items  Late_Risk_Rate
          Fan Shop 1.711387e+07 1.834155e+06        66861       54.774831
           Apparel 7.976255e+06 8.818829e+05        48998       54.747133
              Golf 4.609028e+06 4.975236e+05        33220       54.780253
          Footwear 4.006499e+06 4.102225e+05        14525       54.726334
          Outdoors 1.253351e+06 1.452515e+05         9686       55.492463
        Technology 1.039599e+06 1.131700e+05         1465       55.017065
           Fitness 3.970509e+05 4.653806e+04         2479       55.546591
        Discs Shop 2.288877e+05 2.419312e+04         2026       54.442251
Health and Beauty  1.060805e+05 9.493630e+03          362       55.801105
          Pet Shop 4.152480e+04 3.589260e+03          492       58.943089
         Book Shop 1.258740e+04 8.830100e+02          405       56.543210


In [14]:
# ============================================
# 13. BUSINESS VALUE AT DELIVERY RISK
# ============================================

risk_exposure = df.groupby("Late_delivery_risk").agg(
    Sales=("Sales", "sum"),
    Profit=("Order Profit Per Order", "sum"),
    Order_Items=("Order Item Id", "count")
).reset_index()

risk_exposure["Risk_Label"] = risk_exposure["Late_delivery_risk"].map({
    0: "No Late Risk",
    1: "Late Risk"
})

risk_exposure["Sales_Percentage"] = (
    risk_exposure["Sales"] /
    risk_exposure["Sales"].sum() * 100
)

risk_exposure["Profit_Percentage"] = (
    risk_exposure["Profit"] /
    risk_exposure["Profit"].sum() * 100
)

print(
    risk_exposure[
        ["Risk_Label", "Sales", "Profit",
         "Order_Items", "Sales_Percentage",
         "Profit_Percentage"]
    ].to_string(index=False)
)

  Risk_Label        Sales       Profit  Order_Items  Sales_Percentage  Profit_Percentage
No Late Risk 1.665834e+07 1.826851e+06        81542         45.286013          46.052331
   Late Risk 2.012640e+07 2.140052e+06        98977         54.713987          53.947669


In [15]:
  Risk_Label        Sales       Profit  Order_Items  Sales_Percentage  Profit_Percentage
No Late Risk 1.665834e+07 1.826851e+06        81542         45.286013          46.052331
   Late Risk 2.012640e+07 2.140052e+06        98977         54.713987          53.947669

IndentationError: unexpected indent (3656816117.py, line 1)

In [16]:
# ============================================
# 14. MARKET PERFORMANCE & DELIVERY RISK
# ============================================

market_analysis = df.groupby("Market").agg(
    Sales=("Sales", "sum"),
    Profit=("Order Profit Per Order", "sum"),
    Order_Items=("Order Item Id", "count"),
    Late_Risk=("Late_delivery_risk", "sum")
).reset_index()

market_analysis["Late_Risk_Rate"] = (
    market_analysis["Late_Risk"] /
    market_analysis["Order_Items"] * 100
)

market_analysis["Sales_Risk_Exposure"] = (
    df[df["Late_delivery_risk"] == 1]
    .groupby("Market")["Sales"]
    .sum()
    .reindex(market_analysis["Market"])
    .fillna(0)
    .values
)

market_analysis = market_analysis.sort_values(
    "Late_Risk_Rate",
    ascending=False
)

print(
    market_analysis[
        ["Market", "Sales", "Profit",
         "Order_Items", "Late_Risk_Rate",
         "Sales_Risk_Exposure"]
    ].to_string(index=False)
)

      Market        Sales       Profit  Order_Items  Late_Risk_Rate  Sales_Risk_Exposure
      Europe 1.087240e+07 1.169443e+06        50252       55.207753         5.979287e+06
Pacific Asia 8.273744e+06 8.577534e+05        41260       55.046049         4.552200e+06
        USCA 5.066529e+06 5.643138e+05        25799       54.800574         2.757674e+06
      Africa 2.294453e+06 2.520712e+05        11614       54.589289         1.248052e+06
       LATAM 1.027761e+07 1.123322e+06        51594       54.355158         5.589183e+06
